# Insurance Claims Analytics & Risk Assessment
## Module 1 — Data Preparation

**Business Objective**
Establish a clean, validated, and feature-enriched analytical dataset as the
single source of truth for all downstream analysis. Ensuring data integrity
at this stage prevents propagation of errors into statistical tests, models,
and business recommendations.

**Dataset Overview**
| Attribute | Detail |
|---|---|
| Source | InsuranceData.csv |
| Raw Records | 10,004 |
| Features | 13 |
| Date Range | 2023–2025 |

**Analysis Scope**
Schema validation · Duplicate detection · Date parsing · Categorical
standardisation · Missing value treatment · Feature engineering


In [ ]:
import sys
sys.path.insert(0, '..')
from src.data_preparation import (
    load_raw_data, validate_schema, audit_data_quality,
    parse_dates, standardise_categoricals, handle_missing_values,
    engineer_features, save_cleaned_data
)
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


## 1.1 Raw Data Ingestion

The dataset is loaded and immediately subjected to schema validation to confirm
all expected columns are present before any processing begins.

In [ ]:
df_raw = load_raw_data()
print(f"Shape: {df_raw.shape}")
df_raw.head()


## 1.2 Data Quality Audit

A structured audit captures null rates, data types, and unique value counts
across all columns. This informs the cleaning strategy applied downstream.

In [ ]:
validate_schema(df_raw)
audit = audit_data_quality(df_raw)
audit


**Key Observations**
- `ClaimDate` carries 43.5% missing values — expected behaviour. Policies with no filed claim have no associated claim date.
- No missing values in financial columns (`PremiumAmount`, `CoverageAmount`, `ClaimAmount`).
- 4 duplicate rows detected and will be addressed during cleaning.


## 1.3 Cleaning & Standardisation

In [ ]:
df = parse_dates(df_raw.copy())
df = standardise_categoricals(df)
df = handle_missing_values(df)
print("Cleaning complete.")
print(f"ClaimDate nulls retained (no claim filed): {df['ClaimDate'].isna().sum()}")


## 1.4 Feature Engineering

Derived features are constructed to support segmentation, anomaly detection,
and predictive modelling. Each feature has a defined analytical purpose.

| Feature | Formula | Purpose |
|---|---|---|
| `AgeGroup` | Binned Age | Demographic segmentation |
| `PolicyDuration` | EndDate − StartDate | Policy term validation |
| `Days_to_Claim` | ClaimDate − StartDate | Lifecycle timing |
| `ClaimTiming` | Bucketed Days_to_Claim | Early/Mid/Late classification |
| `Coverage_Ratio` | Coverage / Premium | Value-for-money indicator |
| `Utilization` | Claim / Coverage | Financial exposure rate |
| `Claim_Flag` | ClaimAmount > 0 | Binary claim indicator |
| `Early_Claim_Flag` | Days_to_Claim ≤ 30 | Early claim risk marker |


In [ ]:
df = engineer_features(df)
save_cleaned_data(df)
print(f"Final dataset shape: {df.shape}")
df[['Age', 'AgeGroup', 'Coverage_Ratio', 'Utilization', 'Days_to_Claim', 'ClaimTiming', 'Claim_Flag', 'Early_Claim_Flag']].head(10)


## 1.5 Final Dataset Summary


In [ ]:
print("=== Cleaned Dataset Overview ===")
print(f"Total records    : {len(df):,}")
print(f"Total features   : {df.shape[1]}")
print(f"\nClaim Status Distribution:")
print(df['ClaimStatus'].value_counts().to_string())
print(f"\nPolicy Type Distribution:")
print(df['PolicyType'].value_counts().to_string())
print(f"\nAge Group Distribution:")
print(df['AgeGroup'].value_counts().sort_index().to_string())


---
**Next Module →** `02_EDA.ipynb` — Exploratory Data Analysis
